# Bulldozer Price Prediction Project

Predicting the sale price of bulldozers using machine learning.

In this notebook, we're going to go through an example machine learning
project with the goal of predicting the sale price of bulldozers.


## 1. Problem definition

> How well can we predict the future sales price of a bulldozer,
> given its characteristics and previous examples of the past
> sales price of similar bulldozers.

## 2 Data

The data is downloaded from the Kaggle Bluebook for Bulldozers competition.

From [the website](https://www.kaggle.com/c/bluebook-for-bulldozers/data):

The data for this competition is split into three parts:

- Train.csv is the training set, which contains data through the end of 2011.
- Valid.csv is the validation set, which contains data from
January 1, 2012 - April 30, 2012 You make predictions on this set
throughout the majority of the competition. Your score on this set is used
to create the public leaderboard.
- Test.csv is the test set, which won't be released until the last week
of the competition. It contains data from May 1, 2012 - November 2012.
Your score on the test set determines your final rank for the competition.

The key fields are in train.csv are:

- SalesID: the uniue identifier of the sale
- MachineID: the unique identifier of a machine.  A machine can be
sold multiple times
- saleprice: what the machine sold for at auction (only provided
in train.csv)
- saledate: the date of the sale

There are several fields towards the end of the file on the different
options a machine can have.  The descriptions all start with
"machine configuration" in the data dictionary.  Some product types
do not have a particular option, so all the records for that option
variable will be null for that product type.  Also, some sources do not
provide good option and/or hours data.

The machine_appendix.csv file contains the correct year manufactured
for a given machine along with the make, model, and product class
details. There is one machine id for every machine in all the
competition datasets (training, evaluation, etc.).


## 3. Evaluation

Again, from
[the Kaggle competition evaluation](www.kaggle.com/competitions/bluebook-for-bulldozers/overview/evaluation).

The evaluation metric for this competition is the RMSLE (room mean
squared log error) between the actual and predicted auction prices.

For more for the evaluation of this project check the Kaggle evaluation
section (see above).

**Note**: The goal for most regression evaluation metrics is to minimize
the error. For example, our goal for this project is to build a machine
learning model which minimizes RMSLE.

## 4. Features

Kaggle provides a data dictionary detailing all the features of the
data set. You can view this data dictionary using Excel, Mac Numbers,
Google Sheets, or even using PyCharm.


In [ ]:
# Import required packages

# Import cytoolz
import cytoolz.curried as ctc

# Import data analysis packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import machine learning packages
import sklearn

In [ ]:
# Create a `default_rng` for `numpy`
rng = np.random.default_rng(42)

In [ ]:
# Load training **and** validation sets.
# Reason will be clarified in later video.
df = pd.read_csv('data/bluebook-for-bulldozers/TrainAndValid.csv')

Daniel encountered the same warning in the video. His solution:
set `low_memory=False`.

In [ ]:
del df

In [ ]:
df = pd.read_csv(
    'data/bluebook-for-bulldozers/TrainAndValid.csv',
    low_memory=False, # Tells pandas to **not** try to minimize space
)

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
# Let's plot some columns
fig, ax = plt.subplots()
ax.scatter(df['saledate'], df['SalePrice'])
plt.show()

Hmmm. Plotting all the data is **not** very useful - just as Daniel
said in the video. :)

In [ ]:
# Let's plot a subset of the data
fig, ax = plt.subplots()
ax.scatter(df['saledate'][:1000], df['SalePrice'][:1000])
plt.show()

In [ ]:
df.saledate[:1000]

A scatter plot, even of only a portion of the data, does not seem
very useful either.  Let's try plotting a histogram.

In [ ]:
# Always a good idea to plot our target variable(s)
df['SalePrice'].plot.hist()
plt.show()

### Parsing dates

When we look at timeseries data, we want to "enrich" the time and date
component as much as possible.

We can accomplish this by telling `pandas` which of our columns contain
dates using the `parse_dates` parameter.

In [ ]:
# Notice the previous `dtype`
df.saledate.dtype

In [ ]:
# Import data again but this time parse dates
df = pd.read_csv(
    'data/bluebook-for-bulldozers/TrainAndValid.csv',
    low_memory=False,
    parse_dates=['saledate'],
)

In [ ]:
df.saledate.dtype

In [ ]:
df['saledate'][:1000]

In [ ]:
# Let's plot our limited scatter plot again
fig, ax = plt.subplots()
ax.scatter(df['saledate'][:1000], df['SalePrice'][:1000])
plt.show()

In [ ]:
df.head()

In [ ]:
# A trick for viewing all our columns
df.head().T

In [ ]:
# Let's remind ourselves of our sales data
df.saledate.head(20)

## Sort `DataFrame` by `saledate`

When working with time series data, it is a good practice to sort
the data by the time point of interest. (In this case, `saledate`).

In [ ]:
# Sort `DataFrame` in date order
df.sort_values(by=['saledate'], ascending=True, inplace=True)
df.saledate.head(20)

In [ ]:
df.head()

In [ ]:
# Make a copy
df_tmp = df.copy()
df_tmp

### Add datetime parameters to `saledate` column


In [ ]:
df.columns

In [ ]:
df_tmp[:1].saledate.dt.year

In [ ]:
df_tmp[:1].saledate.dt.day

In [ ]:
df_tmp[:1].saledate

In [ ]:
df_tmp['saleyear'] = df_tmp['saledate'].dt.year
df_tmp['salemonth'] = df_tmp.saledate.dt.month
df_tmp['saleday'] = df_tmp.saledate.dt.day
df_tmp['saledayofweek'] = df_tmp.saledate.dt.dayofweek
df_tmp['saledayofyear'] = df_tmp.saledate.dt.dayofyear


In [ ]:
df_tmp.head().T.tail()

In [ ]:
# Now that we've enriched our `DataFrame` with date time features,
# we can remove the `saledate` column (from `df_tmp`)
df_tmp.drop('saledate', axis=1, inplace=True)

In [ ]:
'saledate' in df_tmp.columns

In [ ]:
# Which state has the most soles?
df_tmp.state.value_counts()

In [ ]:
len(df_tmp)

## 5. Modeling

We've done enough exploratory data analysis (EDA - we could always
do more). Let's start modelling! More specifically, let's do some
_model-driven EDA_.

What kind of learning should we do?

Back to our [Online machine learning model map](https://scikit-learn.org/stable/machine_learning_map.html).

Walking through the map and primarily because we have experience
with this learning, lets build a `RandomForestRegressor`

In [ ]:
# Let's build a machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_jobs=-1, # use all available processors
    random_state=rng.integers(np.iinfo(np.int32).max),
)

# Our target is 'SalePrice' so we drop it from the data
# **before** fitting and supply that column as the target
# column for fitting our model.
#
# In our X and y nomenclature, X is all the columns but `SalePrice`
# and y is the `SalePrice` column.
try:
    model.fit(df_tmp.drop('SalePrice', axis=1), df_tmp['SalePrice'])
except ValueError as ve:
    print(ve)

In [ ]:
df.info()

In [ ]:
# Let's look at one of the columns of type `object`
df_tmp['UsageBand'].head()

### Convert strings to categories

One way to turn all of data into numbers is to convert them to pandas
`Categorical` instances.

Pandas `Categorical` types

- [Categorical API](https://pandas.pydata.org/docs/reference/arrays.html#categoricals
- [Categorical data user guide](https://pandas.pydata.org/docs/user_guide/categorical.html)

Additionally, see [Pandas arrays, scalars and data types](https://pandas.pydata.org/docs/reference/arrays.html#pandas-arrays-scalars-and-data-types)

Machine learning is about

- Getting data
- "Massaging" the data into a format amenable to a machine learning moel
- Find patterns
- Evaluating how well our steps worked


In [ ]:
df_tmp.head().T

In [ ]:
# Let's investigate
pd.api.types.is_string_dtype(df_tmp['UsageBand'])

In [ ]:
df_tmp['UsageBand']

In [ ]:
df_tmp[df_tmp['UsageBand'].notnull()]['UsageBand']

Let's investigate the `UsageBand` column
- `pd.api.types.is_string_dtype(df_tmp['UsageBand'])`
- Hmm...
	- In the video, this function returned `True`
	- However, in my environment, this function returns `False`

This text summarizes the changes to pandas function `is_string_dtype()`.
It is copied from the experimental AI Overview generated on
2024-01-28.

The behavior of pandas.api.types.is_string_dtype has changed in recent
versions of pandas, particularly with the introduction of the dedicated
string dtype in pandas 2.0 and its enhancements in pandas 2.2.

Here's a breakdown of the changes:

**Pandas 2.0 and earlier**:

- `is_string_dtype` would return `True` for object dtype columns if
all the elements in the column were strings.
- It would return `False` for columns with mixed data types or
non-string objects.

**Pandas 2.2 and later:**

- Pandas introduced a new string dtype (`string`) backed by Arrow,
which provides significant performance and memory improvements.
- `is_string_dtype` now returns `True` for columns with the new string dtype.
- It still returns `True` for object dtype columns if all elements
are strings, but this behavior is considered deprecated.

**Pandas 3.0 and later:**

- The default string dtype will be the Arrow-backed string dtype,
and `is_string_dtype` will primarily be used to identify columns with
this dtype.
- PyArrow will become a required dependency for pandas.

**Key points to remember:**

- If you're using pandas 2.2 or later, consider enabling the new
string dtype inference with `pd.options.future.infer_string = True`.
- If you're relying on `is_string_dtype` to check for object `dtype`
columns containing only strings, be aware that this behavior is deprecated
and may change in the future.
- For better performance and memory efficiency, use the new string dtype when working with string data in pandas.

Generative AI is experimental.

In [ ]:
# find the columns which contain strings
# for label, content in df_tmp.items():
#     if pd.api.types.is_string_dtype(content):
#         print(label)
for label in df_tmp.columns:
    if df_tmp[label].dtype == 'object':
        try:
            df_tmp[label].astype(str)
            print(label)
        except ValueError:
            print(f'Cannot convert column, {label}, to string.')

In [ ]:
# If you wondering what df.items() does, here's an example.
random_dict = {'key1': 'hello',
               'key2': 'world!'}

for key, value in random_dict.items():
    print(f'this is a key: {key}',
          f', this is a value: {value}')

In [ ]:
# I think I can make this code work, but a bit too much time right now.
# ctc.pipe(
#     df_tmp.columns,
#     ctc.filter(lambda cn: df_tmp[cn].dtype == 'object'),
#     ctc.map(lambda cn: df_tmp[cn].map(str)),
#     ctc.filter(lambda cn: cn == 'UsageBand'),
#     pd.Series,
# )

In [ ]:
columns_to_convert = [label for
                      label in df_tmp.columns
                      if df_tmp[label].dtype == 'object']
for label in columns_to_convert:
    df_tmp[label] = df_tmp[label].astype('string')

In [ ]:
df_tmp[df_tmp['UsageBand'].notnull()]['UsageBand']

In [ ]:
# And now back to our regularly scheduled video. :)
# This will turn all of the string values into category values
for label, content in df_tmp.items():
    if pd.api.types.is_string_dtype(content):
        df_tmp[label] = content.astype('category').cat.as_ordered()

In [ ]:
df_tmp.info()

In [ ]:
df_tmp.state.cat.categories

In [ ]:
df_tmp.state.cat.codes

Thanks to pandas `Categorical` types, we now a way to access all our
data in the form of numbers.

However, we're still **missing some data**.

In [ ]:
# Fraction of missing data for each column
df_tmp.isnull().sum() / len(df_tmp)

In [ ]:
# Export current tmp dataframe (a checkpoint of our work)
df_tmp.to_csv('./data/bluebook-for-bulldozers/train_tmp.csv', index=False)

In [ ]:
# And import the preprocessed data
df_tmp = pd.read_csv('./data/bluebook-for-bulldozers/train_tmp.csv',
                     low_memory=False)
df_tmp.head().T

In [ ]:
# Remind ourselves that we still have **missing** values
df_tmp.isna().sum()